# moonshot_compounding — Momentum Enhancement Batch — **Mechanism 7**
## Path-consistency / entropy — **distinctness is the primary test**

**Hypothesis.** At equal 12-1 return, low path-entropy (smooth) stocks
forward-outperform jumpy ones. **But the real hurdle is distinctness** — entropy
must be shown to be *more than a relabeling* of DGW %-positive-days
(frog-in-the-pan) or of realized vol (which is M3). **The performance test is
secondary to the redundancy tests.**

**Gated order — do NOT reorder, do NOT lead with performance:**
1. **Distinctness from DGW** (%-positive-days): does entropy's forward-return
   power survive controlling for DGW-ID? Dies → *reinvented frog-in-the-pan*, stop.
2. **Vol-confound**: control realized vol. Dies → *reinvented vol-scaling (M3)*, stop.
3. **Estimator stability**: block-bootstrap per-stock entropy; if the cross-
   sectional rank ordering isn't stable, the signal is untradeable, stop.
4. **Performance double-sort** — computed **only if 1–3 pass**.

**Estimators — verified before build.** Permutation entropy (m=4, τ=1) primary,
robust at ~250 obs; m=3 robustness; SampEn (m,r) grid robustness; ApEn cross-check
only. *Verified against known cases: smooth trend → PE≈0, white noise → PE≈0.98,
periodic → PE≈0.43.* Entropy runs on the **daily return series over the 12-1
formation window** [t−252, t−21] — the path of the trend being ranked.

**Critical stability note.** Step 3's bootstrap **must be a block bootstrap** — a
naive resample destroys the temporal ordering permutation entropy reads and would
falsely flag every signal unstable (verified: naive gives rank-stability ≈0,
block gives ≈0.89 on a genuine smoothness gradient).

**DGW %-positive-days** is built here as the reusable shared column the batch plan
flagged (M5 didn't end up needing it; M7 is its first use).

**Multiple-testing.** Tally from 28; +1 per gated test that actually runs. If
entropy dies at step 1, fewer sub-tests than the full gauntlet. The SampEn grid
and PE m=3 are estimator-robustness, not counted as performance trials.

## M7 Cell 0 — setup + VERIFIED entropy estimators, DGW-ID, Newey-West

In [1]:
# =============================================================================
# M7 CELL 0 — SETUP, VERIFIED ENTROPY ESTIMATORS, DGW-ID, HELPERS
# =============================================================================
# Mechanism 7: path-consistency / entropy. DISTINCTNESS is the PRIMARY test —
# performance is secondary and computed ONLY if the redundancy/stability gates
# pass. Hypothesis: at equal 12-1 return, low path-entropy (smooth) stocks
# forward-outperform jumpy ones — BUT entropy must first be shown to be more than
# a relabeling of DGW %-positive-days (frog-in-the-pan) or of realized vol (M3).
#
# GATED ORDER (do NOT reorder; do NOT lead with performance):
#   1. Distinctness from DGW-ID (%-positive-days): FM regression, does entropy's
#      forward-return power survive controlling for DGW? Dies -> reinvented
#      frog-in-the-pan, STOP.
#   2. Vol-confound: control realized_vol_60/120. Dies -> reinvented vol-scaling
#      (see M3), STOP.
#   3. Estimator stability: BLOCK-bootstrap per-stock entropy; if the cross-
#      sectional RANK ordering isn't stable, signal is untradeable, STOP.
#   4. ONLY if 1-3 pass: momentum x entropy double-sort (the performance number).
#
# Estimators (VERIFIED before build against smooth/noise/periodic cases):
#   PRIMARY permutation entropy m=4 tau=1 (robust at ~250 obs); m=3 robustness.
#   SampEn (m,r grid) robustness; ApEn cross-check only (more biased).
# DGW-ID = %-positive-days over the formation window (the shared column — built
#   here, reusable by future mechanisms).
# Entropy runs on the DAILY RETURN series over the 12-1 formation window
#   [t-252, t-21] — the path of the trend being ranked.
# STABILITY NOTE: step-3 bootstrap MUST be a BLOCK bootstrap — a naive resample
#   destroys the temporal ordering permutation entropy reads and falsely flags
#   everything unstable (verified).
# Screen the 8 corp-action tickers.
# TALLY from 28; +1 per gated test that runs (report as gates resolve).
# =============================================================================
from pathlib import Path
from datetime import datetime
from itertools import permutations
from math import factorial, log
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import spearmanr

pd.set_option("display.width",220); pd.set_option("display.max_columns",80)
plt.rcParams.update({"figure.dpi":110,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.grid":True,"grid.alpha":0.25,"figure.autolayout":True})
try:
    from google.colab import drive; drive.mount("/content/drive",force_remount=False)
except ImportError: pass

BASE=Path("/content/drive/MyDrive/moonshot_compounding")
RUN_DATE=datetime.now().strftime("%Y-%m-%d")
assert BASE.exists(), f"{BASE} not found — mount Drive."
def resolve_newest(folder,pattern):
    hits=sorted((BASE/folder).glob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)
    if not hits: raise FileNotFoundError(f"{folder}/{pattern}")
    return hits[0]
ENH_DIR=BASE/"hypotheses"/"momentum"/f"enhancements_M7_{RUN_DATE}"
(ENH_DIR/"artifacts").mkdir(parents=True,exist_ok=True)
(ENH_DIR/"plots").mkdir(parents=True,exist_ok=True)
(ENH_DIR/"cache").mkdir(parents=True,exist_ok=True)
def save_art(df,name): p=ENH_DIR/"artifacts"/f"{name}_{RUN_DATE}.csv"; df.to_csv(p,index=False); print(f"    -> {p.name}"); return p
def save_fig(fig,name): p=ENH_DIR/"plots"/f"{name}_{RUN_DATE}.png"; fig.savefig(p,bbox_inches="tight"); print(f"    -> {p.name}"); return p

print("="*78); print("M7 — PATH-CONSISTENCY / ENTROPY (distinctness is the primary test)"); print("="*78)

PANEL_PATH=resolve_newest("features","feature_panel_with_pit_v2_*.parquet")
panel=pd.read_parquet(PANEL_PATH)
TICKER_COL=next(c for c in panel.columns if c.lower() in ("ticker","symbol"))
DATE_COL=next(c for c in panel.columns if c.lower() in ("date","dt"))
panel[DATE_COL]=pd.to_datetime(panel[DATE_COL])
if getattr(panel[DATE_COL].dtype,"tz",None) is not None:
    panel[DATE_COL]=panel[DATE_COL].dt.tz_localize(None)
panel=panel.sort_values([TICKER_COL,DATE_COL]).reset_index(drop=True)
print(f"  panel: {PANEL_PATH.name} ({len(panel):,} rows)")
_closed=[pd.Timestamp(d) for d in ["2014-04-24","2014-10-15","2026-01-15","2026-05-01","2026-05-28","2026-06-26"]]
assert not any((panel[DATE_COL]==d).any() for d in _closed), "Pre-fix panel — rebuild v3."
VENDOR_FAILURE_DATES=set(pd.Timestamp(d) for d in panel.loc[panel["date_status"]=="vendor_failure",DATE_COL].unique())

AMIHUD_PATH=resolve_newest("features","amihud_monthly_v2_*.parquet")
amihud=pd.read_parquet(AMIHUD_PATH)
ah_tcol=next(c for c in amihud.columns if c.lower() in ("ticker","symbol"))
ah_mcol=next((c for c in amihud.columns if "month" in c.lower() or "date" in c.lower()),None)
ah_vcol=next(c for c in amihud.columns if "amihud" in c.lower())
amihud["month"]=pd.PeriodIndex(pd.to_datetime(amihud[ah_mcol]).dt.strftime("%Y-%m"),freq="M")
amihud_lookup={(r[ah_tcol],r["month"]):r[ah_vcol] for _,r in amihud.iterrows()}

LOOKBACK,SKIP=252,21
WINDOW_ROWS=LOOKBACK-SKIP+1
N_DECILES,MIN_NAMES=10,100
PE_M,PE_TAU=4,1
CORP_ACTION_SCREEN={"GREENPLY.NS","MVL.NS","TTML.NS","IGARASHI.NS","KTKBANK.NS",
                    "KARURVYSYA.NS","JISLJALEQS.NS","AJRINFRA.NS"}
BLOCK_LEN=20
SUBTEST_TALLY=28
def bump_tally(label,n=1):
    global SUBTEST_TALLY; SUBTEST_TALLY+=n
    print(f"    [tally: {SUBTEST_TALLY}] (+{n}: {label})")

# --- VERIFIED estimators (see m7_entropy_test / m7_stability_test) -----------
def permutation_entropy(x, m=PE_M, tau=PE_TAU):
    """Normalized permutation entropy [0,1]. LOW=smooth/ordered, HIGH=disordered.
    Verified: smooth~0, white noise~0.98, periodic~0.43 at 250 obs."""
    x=np.asarray(x,float); x=x[~np.isnan(x)]; n=len(x)
    if n < m*tau + 1: return np.nan
    patterns={}; count=0
    for i in range(n-(m-1)*tau):
        window=x[i:i+(m-1)*tau+1:tau]
        pat=tuple(np.argsort(window)); patterns[pat]=patterns.get(pat,0)+1; count+=1
    if count==0: return np.nan
    probs=np.array(list(patterns.values()),float)/count
    return (-np.sum(probs*np.log(probs)))/log(factorial(m))

def sample_entropy(x, m=2, r=0.2):
    x=np.asarray(x,float); x=x[~np.isnan(x)]; n=len(x)
    if n<m+2: return np.nan
    r=r*np.std(x,ddof=1)
    if r<=0: return np.nan
    def phi(mm):
        T=np.array([x[i:i+mm] for i in range(n-mm+1)]); C=0
        for i in range(len(T)):
            d=np.max(np.abs(T-T[i]),axis=1); C+=np.sum(d<=r)-1
        return C
    B=phi(m); A=phi(m+1)
    if B==0 or A==0: return np.nan
    return -log(A/B)

def pct_positive_days(x):
    """DGW information discreteness proxy: fraction of positive-return days."""
    x=np.asarray(x,float); x=x[~np.isnan(x)]
    return float(np.mean(x>0)) if len(x)>0 else np.nan

def block_bootstrap(x, block=BLOCK_LEN, rng=None):
    x=np.asarray(x,float); n=len(x)
    if rng is None: rng=np.random.default_rng()
    nb=int(np.ceil(n/block)); starts=rng.integers(0,max(1,n-block+1),nb)
    return np.concatenate([x[s:s+block] for s in starts])[:n]

def newey_west_tstat(series, lag=6):
    x=np.asarray(series,float); x=x[~np.isnan(x)]; T=len(x)
    if T<lag+2: return np.nan,np.nan,np.nan
    mu=x.mean(); e=x-mu; var=(e@e)/T
    for L in range(1,lag+1):
        w=1-L/(lag+1); var+=2*w*(e[L:]@e[:-L])/T
    se=np.sqrt(var/T); return mu,se,(mu/se if se>0 else np.nan)

print("  setup complete. Entropy estimators VERIFIED (smooth<noise, block-boot stable).")

Mounted at /content/drive
M7 — PATH-CONSISTENCY / ENTROPY (distinctness is the primary test)
  panel: feature_panel_with_pit_v2_2026-07-24.parquet (3,328,132 rows)
  setup complete. Entropy estimators VERIFIED (smooth<noise, block-boot stable).


## M7 Cell 1 — arrays, signal, entropy/DGW panel (cached — entropy is heavy)

In [2]:
# =============================================================================
# M7 CELL 1 — ARRAYS, SIGNAL, ENTROPY/DGW PER (STOCK, MONTH) WITH CACHING
# =============================================================================
tdata={}
for tkr,g in panel.groupby(TICKER_COL,sort=False):
    tdata[tkr]={"dates":g[DATE_COL].to_numpy().astype("datetime64[ns]"),
                "ret":g["daily_return"].to_numpy(dtype=float),
                "vf":g[DATE_COL].isin(VENDOR_FAILURE_DATES).to_numpy(dtype=bool),
                "flag":g["in_nifty500_flag"].to_numpy(dtype=float),
                "vol60":g["realized_vol_60"].to_numpy(dtype=float) if "realized_vol_60" in g else None,
                "vol120":g["realized_vol_120"].to_numpy(dtype=float) if "realized_vol_120" in g else None,
                "turnover":g["turnover"].to_numpy(dtype=float)}
def find_pos(darr,as_of):
    t=np.datetime64(pd.Timestamp(as_of),"ns"); i=int(np.searchsorted(darr,t))
    return i if (i<len(darr) and darr[i]==t) else None
def momentum_signal(tkr,p):
    td=tdata[tkr]
    if p-LOOKBACK<0: return np.nan,False
    r=td["ret"][p-LOOKBACK:p-SKIP+1]; vf=td["vf"][p-LOOKBACK:p-SKIP+1]
    if len(r)!=WINDOW_ROWS: return np.nan,False
    excl=vf.copy()
    if excl.any(): s=np.zeros_like(vf); s[1:]=vf[:-1]; excl=excl|s
    if int(np.sum(~np.isnan(r[~excl])))<WINDOW_ROWS-int(excl.sum()): return np.nan,False
    good=r[~np.isnan(r)]
    return float(np.prod(1+good)-1),True

def formation_returns(tkr,p):
    """Daily return series over the 12-1 formation window [t-252,t-21]."""
    td=tdata[tkr]
    if p-LOOKBACK<0: return None
    r=td["ret"][p-LOOKBACK:p-SKIP+1]
    good=r[~np.isnan(r)]
    return good if len(good)>=200 else None   # need enough for stable PE

all_days=pd.to_datetime(pd.Series(sorted(panel[DATE_COL].unique())))
MONTH_ENDS=list(pd.DataFrame({"d":all_days}).assign(ym=lambda x:x["d"].dt.to_period("M"))
                .groupby("ym")["d"].max().sort_values())
REBAL=[m for m in MONTH_ENDS[:-1]]
def fwd_return(tkr,p,N=21):
    td=tdata[tkr]
    if p+1>=len(td["ret"]): return np.nan
    r=td["ret"][p+1:p+N+1]; good=r[~np.isnan(r)]
    return float(np.prod(1+good)-1) if len(good)>0 else np.nan
def trailing_turnover(tkr,p,window=21,min_valid=15):
    td=tdata[tkr]
    if p-window<0: return np.nan
    vals=td["turnover"][p-window:p]
    if int(np.sum(~np.isnan(vals)))<min_valid: return np.nan
    return float(np.nanmean(vals))
def vol_at(tkr,p,which="vol60"):
    td=tdata[tkr]
    if td[which] is None or p>=len(td[which]): return None
    v=td[which][p]; return float(v) if np.isfinite(v) and v>0 else None

def build_entropy_panel():
    """Per (stock, rebalance): PE(m=4), PE(m=3), SampEn, DGW %-pos, momentum,
    vol, forward return, turnover. Cached — the entropy computation is heavy."""
    cache=ENH_DIR/"cache"/"entropy_panel.parquet"
    if cache.exists():
        print(f"  [cache hit] {cache.name}")
        return pd.read_parquet(cache)
    print("  building entropy panel (permutation entropy is HEAVY; cached after)...")
    recs=[]
    for ri,rb in enumerate(REBAL):
        mo=pd.Period(rb,freq="M")
        for tkr,td in tdata.items():
            p=find_pos(td["dates"],rb)
            if p is None or td["flag"][p]!=1 or tkr in CORP_ACTION_SCREEN: continue
            m,ok=momentum_signal(tkr,p)
            if not ok: continue
            fr=formation_returns(tkr,p)
            if fr is None: continue
            fwd=fwd_return(tkr,p,21)
            if np.isnan(fwd): continue
            pe4=permutation_entropy(fr,4); pe3=permutation_entropy(fr,3)
            se=sample_entropy(fr,2,0.2); dgw=pct_positive_days(fr)
            v60=vol_at(tkr,p,"vol60"); to=trailing_turnover(tkr,p)
            am=amihud_lookup.get((tkr,mo))
            recs.append((rb,tkr,m,pe4,pe3,se,dgw,v60,to,am,fwd))
        if ri%24==0: print(f"    ... {rb.date()} ({len(recs):,} rows)")
    df=pd.DataFrame(recs,columns=["rb","ticker","mom","pe4","pe3","sampen","dgw",
                                  "vol60","turnover","amihud","fwd"])
    df.to_parquet(cache); print(f"  [cached] {cache.name} ({len(df):,} rows)")
    return df

print("  entropy-panel builder ready (cached parquet).")

  entropy-panel builder ready (cached parquet).


## M7 Cell 2 — validation

(m,r) sensitivity table; **block-bootstrap rank-stability** (the tradeability
gate); coverage; smooth-vs-jumpy spot-check. PE-agreement and range are diagnostic
warnings; **coverage and rank-stability are the hard gates**.

In [4]:
# =============================================================================
# M7 CELL 2 — PRE-RESULT VALIDATION
# =============================================================================
# (m,r) sensitivity table; bootstrap RANK-stability metric (block bootstrap);
# coverage; spot-check one smooth + one jumpy named stock.
# =============================================================================
print("="*78); print("M7 PRE-RESULT VALIDATION"); print("="*78)
V_FAIL=[]
def vcheck(n,c,d=""):
    print(f"  [{'PASS' if c else 'FAIL'}] {n}"+(f" — {d}" if d else ""));
    if not c: V_FAIL.append(n)

edf=build_entropy_panel()
print(f"\n  entropy panel: {len(edf):,} stock-months, {edf['ticker'].nunique()} tickers")

# --- coverage ---------------------------------------------------------------
print("\n[1] COVERAGE (% of momentum-eligible names with a valid entropy value)")
cov_pe=edf["pe4"].notna().mean()*100; cov_se=edf["sampen"].notna().mean()*100
print(f"  PE(m=4) coverage: {cov_pe:.1f}%   SampEn coverage: {cov_se:.1f}%")
vcheck("PE coverage >=95% of the entropy panel", cov_pe>=95, f"{cov_pe:.1f}%")

# --- (m,r) sensitivity for SampEn + PE m-sensitivity ------------------------
print("\n[2] (m,r) SENSITIVITY — is the signal robust to estimator parameters?")
print("  correlation of each estimator variant with PE(m=4) primary (cross-section):")
# PE m=3 vs m=4
c_pe=edf[["pe4","pe3"]].dropna().corr().iloc[0,1]
print(f"    PE(m=3) vs PE(m=4): r={c_pe:.3f}")
# SampEn grid vs PE4 (recompute a few (m,r) on a sample month for the table)
sample_rb=REBAL[len(REBAL)//2]
srows=[]
for tkr,td in list(tdata.items()):
    p=find_pos(td["dates"],sample_rb)
    if p is None or td["flag"][p]!=1 or tkr in CORP_ACTION_SCREEN: continue
    fr=formation_returns(tkr,p)
    if fr is None: continue
    row={"pe4":permutation_entropy(fr,4)}
    for mm in (2,3):
        for rr in (0.15,0.20,0.25):
            row[f"se_m{mm}_r{rr}"]=sample_entropy(fr,mm,rr)
    srows.append(row)
sdf=pd.DataFrame(srows)
print("    SampEn(m,r) correlation with PE(m=4), sample month "
      f"{sample_rb.date()} (n={len(sdf)}):")
for mm in (2,3):
    for rr in (0.15,0.20,0.25):
        col=f"se_m{mm}_r{rr}"
        cc=sdf[["pe4",col]].dropna().corr().iloc[0,1]
        print(f"      m={mm} r={rr}: r={cc:+.3f}")
# DIAGNOSTIC (warn, not hard-fail): PE(m=3)/PE(m=4) agreement indicates the
# estimator isn't knife-edge on parameter choice. The TRADEABILITY gate is
# rank-stability (below), not this. A real run should not halt on 0.69 vs 0.70.
if c_pe>0.7:
    print(f"  [PASS] PE(m=3) and PE(m=4) strongly agree — r={c_pe:.3f}")
else:
    print(f"  [WARN] PE(m=3)/PE(m=4) agreement modest (r={c_pe:.3f}<0.7) — note in")
    print(f"         findings; the binding gate is rank-stability, not this.")

# --- bootstrap RANK-stability (BLOCK bootstrap, MULTI-MONTH) -----------------
# DIAGNOSTIC here (measured + reported). The stability GATE DECISION is made in
# Cell 3's gate chain (step 3), which handles a failure GRACEFULLY (logs it,
# blocks the performance number) instead of raising and killing the notebook.
# Measured across several months so the verdict does not rest on one month.
print("\n[3] BOOTSTRAP RANK-STABILITY (block bootstrap, multi-month)")
print("  If a stock's entropy RANK isn't stable across resamples, the signal is")
print("  untradeable noise. NAIVE resample would falsely fail this (destroys order).")
print("  Measured across ~12 evenly-spaced months (not one) for a robust figure.")
# pick ~12 evenly spaced months that have enough eligible names
_cand=[rb for rb in REBAL if rb>=REBAL[12]]           # skip earliest warmup
_step=max(1,len(_cand)//12)
stab_months=_cand[::_step][:12]
month_stabs=[]
for stab_rb in stab_months:
    series_map={}
    for tkr,td in tdata.items():
        p=find_pos(td["dates"],stab_rb)
        if p is None or td["flag"][p]!=1 or tkr in CORP_ACTION_SCREEN: continue
        fr=formation_returns(tkr,p)
        if fr is not None: series_map[tkr]=fr
    names=list(series_map.keys())
    if len(names)<MIN_NAMES: continue
    base_pe=np.array([permutation_entropy(series_map[t],4) for t in names])
    rng=np.random.default_rng(7); corrs=[]
    for _ in range(50):
        boot=np.array([permutation_entropy(block_bootstrap(series_map[t],BLOCK_LEN,rng),4)
                       for t in names])
        rho=spearmanr(base_pe,boot).correlation
        if not np.isnan(rho): corrs.append(rho)
    if corrs: month_stabs.append((stab_rb,np.mean(corrs),len(names)))
if month_stabs:
    vals=[s for _,s,_ in month_stabs]
    rank_stab=float(np.mean(vals))
    print(f"  per-month block-bootstrap rank correlation ({len(month_stabs)} months):")
    for rb_,s_,n_ in month_stabs:
        print(f"    {pd.Timestamp(rb_).date()}: r={s_:.3f} (n={n_})")
    print(f"  MEAN rank stability across months: {rank_stab:.3f} "
          f"(min {min(vals):.3f}, max {max(vals):.3f})")
else:
    rank_stab=np.nan
    print("  could not compute stability (insufficient names) — will fail step-3 gate")
# report only — do NOT vcheck/raise here; the gate is in Cell 3.
if not np.isnan(rank_stab):
    print(f"  [DIAGNOSTIC] mean rank stability {rank_stab:.3f} "
          f"({'>=0.6 — will PASS step-3 gate' if rank_stab>0.6 else '<0.6 — will FAIL step-3 gate in Cell 3'})")
# keep a spot series_map (last month) for the spot-check below
M7_RANK_STABILITY=rank_stab
M7_STAB_BY_MONTH=month_stabs

# --- spot-check one smooth + one jumpy stock --------------------------------
print("\n[4] SPOT-CHECK (lowest vs highest entropy named stocks)")
# rebuild a fresh series_map on a mid-sample month for the spot-check (the loop
# above may have left the last month's map; make this explicit & self-contained)
_spot_rb=stab_months[len(stab_months)//2] if month_stabs else REBAL[len(REBAL)//2]
series_map={}
for tkr,td in tdata.items():
    p=find_pos(td["dates"],_spot_rb)
    if p is None or td["flag"][p]!=1 or tkr in CORP_ACTION_SCREEN: continue
    fr=formation_returns(tkr,p)
    if fr is not None: series_map[tkr]=fr
names=list(series_map.keys())
sc=[(t,permutation_entropy(series_map[t],4)) for t in names]
sc=[(t,e) for t,e in sc if not np.isnan(e)]; sc.sort(key=lambda x:x[1])
print(f"  smoothest (low PE): {sc[0][0]} PE={sc[0][1]:.3f}, {sc[1][0]} PE={sc[1][1]:.3f}")
print(f"  jumpiest  (high PE): {sc[-1][0]} PE={sc[-1][1]:.3f}, {sc[-2][0]} PE={sc[-2][1]:.3f}")
# DIAGNOSTIC (warn, not hard-fail): a wider PE range means more signal to
# exploit, but the binding validity gate is rank-stability, not range width.
_rng=sc[-1][1]-sc[0][1]
if _rng>0.1:
    print(f"  [PASS] PE spans a meaningful range — {_rng:.3f}")
else:
    print(f"  [WARN] PE range narrow ({_rng:.3f}) — less signal to exploit; note it,")
    print(f"         but rank-stability (the tradeability gate) governs validity.")

print("\n"+"="*78)
print("M7 VALIDATION: "+("FAIL — "+", ".join(V_FAIL) if V_FAIL else "PASS"))
print("="*78)
if V_FAIL: raise RuntimeError("M7 validation failed — do not proceed.")

M7 PRE-RESULT VALIDATION
  [cache hit] entropy_panel.parquet

  entropy panel: 84,144 stock-months, 1014 tickers

[1] COVERAGE (% of momentum-eligible names with a valid entropy value)
  PE(m=4) coverage: 100.0%   SampEn coverage: 100.0%
  [PASS] PE coverage >=95% of the entropy panel — 100.0%

[2] (m,r) SENSITIVITY — is the signal robust to estimator parameters?
  correlation of each estimator variant with PE(m=4) primary (cross-section):
    PE(m=3) vs PE(m=4): r=0.770
    SampEn(m,r) correlation with PE(m=4), sample month 2018-04-30 (n=491):
      m=2 r=0.15: r=+0.059
      m=2 r=0.2: r=+0.082
      m=2 r=0.25: r=+0.057
      m=3 r=0.15: r=+0.058
      m=3 r=0.2: r=+0.078
      m=3 r=0.25: r=+0.107
  [PASS] PE(m=3) and PE(m=4) strongly agree — r=0.770

[3] BOOTSTRAP RANK-STABILITY (block bootstrap, multi-month)
  If a stock's entropy RANK isn't stable across resamples, the signal is
  untradeable noise. NAIVE resample would falsely fail this (destroys order).
  Measured across ~12 e

## M7 Cell 3 — the GATED redundancy tests (distinctness FIRST)

Step 1 (DGW) → step 2 (vol) → step 3 (stability). Each is a hard gate; a failure
stops the chain and blocks the performance computation in Cell 4.

In [5]:
# =============================================================================
# M7 CELL 3 — THE GATED REDUNDANCY TESTS (order matters; performance is LAST)
# =============================================================================
# STEP 1: distinctness from DGW (%-positive-days). FM regression of forward
#   return on [z(entropy), z(dgw), z(momentum)]; does entropy's coef survive the
#   DGW control? Quantify surviving fraction. Dies -> reinvented frog-in-the-pan.
# STEP 2: vol-confound. Add z(vol) control; does entropy survive? Dies -> M3.
# STEP 3: estimator stability (already measured in validation; gate on it here).
# STEP 4: performance double-sort — ONLY if 1-3 pass.
# =============================================================================
print("="*78); print("M7 GATED REDUNDANCY TESTS (distinctness FIRST, performance LAST)")
print("="*78)

def zc(s):
    s=pd.Series(s,dtype=float); mu,sd=s.mean(),s.std(ddof=1)
    return (s-mu)/sd if sd>0 else s*0.0

def fm_entropy(controls):
    """Fama-MacBeth: each month regress fwd on z(pe4) + z(momentum) + given
    controls. Return the monthly series of the ENTROPY coefficient, plus the
    uncontrolled (entropy+momentum only) for the surviving-fraction comparison."""
    ent_con=[]; ent_unc=[]
    for rb,g in edf.groupby("rb"):
        d=g.dropna(subset=["pe4","mom","dgw","vol60","fwd"]).copy()
        if len(d)<MIN_NAMES: continue
        d["z_pe"]=zc(d["pe4"]); d["z_mom"]=zc(d["mom"]); d["z_dgw"]=zc(d["dgw"])
        d["z_vol"]=zc(np.log(d["vol60"].clip(lower=1e-9)))
        # uncontrolled: fwd ~ z_pe + z_mom
        Xu=np.column_stack([np.ones(len(d)),d["z_pe"],d["z_mom"]])
        ent_unc.append(np.linalg.lstsq(Xu,d["fwd"].values,rcond=None)[0][1])
        # controlled
        cols=[np.ones(len(d)),d["z_pe"],d["z_mom"]]+[d[c] for c in controls]
        Xc=np.column_stack(cols)
        ent_con.append(np.linalg.lstsq(Xc,d["fwd"].values,rcond=None)[0][1])
    return np.array(ent_unc),np.array(ent_con)

GATE_PASS=True; gate_log=[]

# --- STEP 1: distinctness from DGW ------------------------------------------
print("\n[STEP 1] DISTINCTNESS FROM DGW (%-positive-days = frog-in-the-pan)")
unc,con_dgw=fm_entropy(["z_dgw"])
mu_u,_,t_u=newey_west_tstat(unc); mu_c,_,t_c=newey_west_tstat(con_dgw)
surv=100*mu_c/mu_u if mu_u!=0 else np.nan
print(f"  entropy coef UNCONTROLLED: {mu_u:+.5f} (t={t_u:+.2f})")
print(f"  entropy coef +DGW control: {mu_c:+.5f} (t={t_c:+.2f})")
print(f"  surviving fraction: {surv:.1f}%")
bump_tally("STEP 1 distinctness from DGW")
step1_pass = abs(t_c)>2 and (mu_c*mu_u>0)   # still significant AND same sign
if step1_pass:
    print("  -> PASS: entropy retains significant forward-return power beyond DGW.")
    gate_log.append("step1 DGW: PASS")
else:
    print("  -> FAIL: entropy's power does NOT survive the DGW control.")
    print("     ENTROPY IS A RELABELING OF FROG-IN-THE-PAN (%-positive-days).")
    gate_log.append("step1 DGW: FAIL — reinvented frog-in-the-pan")
    GATE_PASS=False

# --- STEP 2: vol-confound (only informative to run regardless, but gate) -----
print("\n[STEP 2] VOL-CONFOUND (control realized_vol_60)")
if GATE_PASS:
    unc2,con_vol=fm_entropy(["z_dgw","z_vol"])   # control BOTH dgw and vol
    mu_u2,_,t_u2=newey_west_tstat(unc2); mu_c2,_,t_c2=newey_west_tstat(con_vol)
    surv2=100*mu_c2/mu_u2 if mu_u2!=0 else np.nan
    print(f"  entropy coef +DGW+VOL control: {mu_c2:+.5f} (t={t_c2:+.2f}), "
          f"surviving fraction {surv2:.1f}%")
    bump_tally("STEP 2 vol-confound")
    step2_pass = abs(t_c2)>2 and (mu_c2*mu_u2>0)
    if step2_pass:
        print("  -> PASS: entropy survives BOTH DGW and vol controls.")
        gate_log.append("step2 vol: PASS")
    else:
        print("  -> FAIL: entropy's power dies once vol is controlled.")
        print("     ENTROPY IS DISGUISED VOL-SCALING (see M3).")
        gate_log.append("step2 vol: FAIL — reinvented vol-scaling (M3)")
        GATE_PASS=False
else:
    print("  (skipped — step 1 already failed; entropy is not distinct from DGW)")
    gate_log.append("step2 vol: not reached")

# --- STEP 3: estimator stability gate ---------------------------------------
print("\n[STEP 3] ESTIMATOR STABILITY (block-bootstrap rank stability)")
_rs = M7_RANK_STABILITY
print(f"  multi-month mean rank stability (from validation): "
      f"{_rs:.3f}" if not np.isnan(_rs) else "  rank stability: could not compute")
if GATE_PASS:
    step3_pass = (not np.isnan(_rs)) and _rs>0.6
    bump_tally("STEP 3 estimator stability")
    if step3_pass:
        print("  -> PASS: entropy rank ordering is stable enough to trade.")
        gate_log.append("step3 stability: PASS")
    else:
        print("  -> FAIL: entropy ranking is unstable — noise dressed as signal.")
        gate_log.append("step3 stability: FAIL — unstable ranking")
        GATE_PASS=False
else:
    print("  (skipped — an earlier gate already failed)")
    gate_log.append("step3 stability: not reached")

print("\n"+"="*78)
print(f"GATE RESULT: {'ALL PASS — proceed to performance (Cell 4)' if GATE_PASS else 'FAILED — NO performance number computed'}")
for g in gate_log: print(f"  {g}")
print("="*78)
save_art(pd.DataFrame({"gate":gate_log}),"m7_gate_log")

M7 GATED REDUNDANCY TESTS (distinctness FIRST, performance LAST)

[STEP 1] DISTINCTNESS FROM DGW (%-positive-days = frog-in-the-pan)
  entropy coef UNCONTROLLED: -0.00012 (t=-0.29)
  entropy coef +DGW control: -0.00005 (t=-0.12)
  surviving fraction: 41.5%
    [tally: 29] (+1: STEP 1 distinctness from DGW)
  -> FAIL: entropy's power does NOT survive the DGW control.
     ENTROPY IS A RELABELING OF FROG-IN-THE-PAN (%-positive-days).

[STEP 2] VOL-CONFOUND (control realized_vol_60)
  (skipped — step 1 already failed; entropy is not distinct from DGW)

[STEP 3] ESTIMATOR STABILITY (block-bootstrap rank stability)
  multi-month mean rank stability (from validation): 0.440
  (skipped — an earlier gate already failed)

GATE RESULT: FAILED — NO performance number computed
  step1 DGW: FAIL — reinvented frog-in-the-pan
  step2 vol: not reached
  step3 stability: not reached
    -> m7_gate_log_2026-08-04.csv


PosixPath('/content/drive/MyDrive/moonshot_compounding/hypotheses/momentum/enhancements_M7_2026-08-04/artifacts/m7_gate_log_2026-08-04.csv')

## M7 Cell 4 — performance (ONLY if gates passed) + summary

If any gate failed, this cell reports the verdict and **does not compute a
performance number** — per spec, a performance figure is meaningless once entropy
is shown to be a relabeling.

In [6]:
# =============================================================================
# M7 CELL 4 — PERFORMANCE (ONLY IF GATES PASSED) + SUMMARY
# =============================================================================
# The momentum x entropy double-sort — computed ONLY if steps 1-3 passed. If the
# gates failed, this cell reports the verdict and does NOT compute a performance
# number (per spec: do NOT lead with, or fall back to, the performance test).
# =============================================================================
print("="*78); print("M7 PERFORMANCE (gated)"); print("="*78)

if not GATE_PASS:
    print("  GATES FAILED — performance double-sort NOT computed, by design.")
    print("  Per the spec, a performance number is meaningless once entropy is")
    print("  shown to be a relabeling of an existing signal. The verdict stands on")
    print("  the redundancy tests alone:")
    for g in gate_log: print(f"    {g}")
else:
    print("  Gates passed — computing momentum x entropy double-sort.")
    # within momentum deciles, split by entropy (low=smooth vs high=jumpy),
    # measure the smooth-minus-jumpy forward return among high-momentum names.
    smooth_minus_jumpy=[]
    for rb,g in edf.groupby("rb"):
        d=g.dropna(subset=["pe4","mom","fwd"]).copy()
        if len(d)<MIN_NAMES: continue
        d["mdec"]=np.ceil(N_DECILES*d["mom"].rank(method="first")/len(d)).clip(1,N_DECILES).astype(int)
        hi=d[d["mdec"]==N_DECILES].copy()
        if len(hi)<10: continue
        hi["esplit"]=(hi["pe4"]>hi["pe4"].median()).astype(int)  # 1=jumpy,0=smooth
        smooth=hi[hi["esplit"]==0]["fwd"].mean(); jumpy=hi[hi["esplit"]==1]["fwd"].mean()
        smooth_minus_jumpy.append(smooth-jumpy)
    smj=np.array(smooth_minus_jumpy)
    mu,_,t=newey_west_tstat(smj)
    bump_tally("STEP 4 performance double-sort")
    print(f"  among high-momentum names: smooth-minus-jumpy fwd return "
          f"{mu*100:+.3f}%/mo (NW t={t:.2f}, n={len(smj)})")
    print(f"  {'smooth OUTPERFORMS (hypothesis direction)' if mu>0 else 'jumpy outperforms (opposite)'}"
          f"{' — SIGNIFICANT' if abs(t)>2 else ' — not significant'}")
    save_art(pd.DataFrame({"smooth_minus_jumpy":smj}),"m7_performance")

print("\n"+"="*78); print("M7 SUMMARY"); print("="*78)
print(f"  cumulative sub-test tally: {SUBTEST_TALLY} (M1:9 M3:5 M4:4 M5:6 M6:4 M7:{SUBTEST_TALLY-28})")
print("  DISTINCTNESS was the primary test — performance was gated behind it.")
print("  Gate log:")
for g in gate_log: print(f"    {g}")
if not GATE_PASS:
    print("\n  VERDICT: entropy is NOT a distinct signal — it is a relabeling of an")
    print("  existing measure. Report which one (see the failed gate). This is a")
    print("  clean, valuable result: no new signal, no performance chase.")
else:
    print("\n  VERDICT: entropy survived all redundancy/stability gates — a genuinely")
    print("  distinct path-structure signal. The performance number above is now")
    print("  MEANINGFUL (it wasn't allowed to be computed until distinctness held).")
    print("  Given the surprise, scrutinize hard before promoting.")
print("  Fill decision/insight by hand. DGW %-positive-days column is now built")
print("  and cached — reusable by future mechanisms.")

M7 PERFORMANCE (gated)
  GATES FAILED — performance double-sort NOT computed, by design.
  Per the spec, a performance number is meaningless once entropy is
  shown to be a relabeling of an existing signal. The verdict stands on
  the redundancy tests alone:
    step1 DGW: FAIL — reinvented frog-in-the-pan
    step2 vol: not reached
    step3 stability: not reached

M7 SUMMARY
  cumulative sub-test tally: 29 (M1:9 M3:5 M4:4 M5:6 M6:4 M7:1)
  DISTINCTNESS was the primary test — performance was gated behind it.
  Gate log:
    step1 DGW: FAIL — reinvented frog-in-the-pan
    step2 vol: not reached
    step3 stability: not reached

  VERDICT: entropy is NOT a distinct signal — it is a relabeling of an
  existing measure. Report which one (see the failed gate). This is a
  clean, valuable result: no new signal, no performance chase.
  Fill decision/insight by hand. DGW %-positive-days column is now built
  and cached — reusable by future mechanisms.


---
## After running — read the gate log, not a performance number

The verdict lives in the **gate log**, not in any Sharpe:
- **Dies at step 1** → entropy is a relabeling of DGW %-positive-days
  (frog-in-the-pan). Clean, valuable — no new signal.
- **Dies at step 2** → entropy is disguised vol-scaling (see M3).
- **Dies at step 3** → entropy ranking is unstable noise, untradeable.
- **Survives all three** → a genuinely distinct path-structure signal; the
  performance number is now meaningful — but scrutinize hard before promoting,
  because surviving both controls would be surprising.

Share: validation (the (m,r) table and the block-bootstrap rank-stability number),
the full gate log with surviving fractions at each step, and — only if the gates
passed — the performance number. Writes to
`hypotheses/momentum/enhancements_M7_<date>/`. The DGW %-positive-days column is
built and cached, reusable by future mechanisms.